# 03 — EDA: Bangkok
**Data source:** `data/processed/bangkok_enriched.parquet`  
**Same structure as 02_eda_singapore.ipynb so charts are directly comparable.**

**Sections:**
1. Load & quick overview
2. Price distribution by room type (boxplot) — note: prices in THB
3. Room type breakdown (bar chart)
4. Availability vs occupancy by room type (probability of being used)
5. Neighbourhood — median price ranking
6. Neighbourhood — listing count vs review activity
7. Host portfolio segmentation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DATA = Path('../data/processed')
df   = pd.read_parquet(DATA / 'bangkok_enriched.parquet')

print(f'Rows: {len(df):,}  |  Cols: {df.shape[1]}')
df.head(3)

---
## 1. Price Distribution by Room Type — Boxplot
**Note:** Bangkok prices are in THB (Thai Baht). Recall from profiling that skewness = 53.24 — extreme outliers exist (max ฿1,000,000).  
We cap at 99th percentile to make the distribution readable.

In [ ]:
p99 = df['price'].quantile(0.99)
plot_df = df[df['price'] <= p99].copy()

room_order = (
    plot_df.groupby('room_type')['price']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=plot_df, x='room_type', y='price',
    order=room_order, palette='Oranges_d', ax=ax
)
ax.set_title('Bangkok — Price Distribution by Room Type (capped at 99th pct)', fontsize=13, pad=12)
ax.set_xlabel('Room Type')
ax.set_ylabel('Price (THB / night)')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('฿{x:,.0f}'))
plt.tight_layout()
plt.show()

print('\nMedian price by room type (THB):')
print(plot_df.groupby('room_type')['price'].median().sort_values(ascending=False).to_string())
print(f'\n99th percentile cutoff: ฿{p99:,.0f} (actual max: ฿{df["price"].max():,.0f})')

---
## 2. Room Type Breakdown — Count Bar Chart
Bangkok skews heavily toward entire homes (65%) vs Singapore (40%). This drives the different pricing premium we observed in model.py Q1.

In [ ]:
room_counts = df['room_type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

room_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('Oranges_d', len(room_counts)), edgecolor='white')
axes[0].set_title('Listing Count by Room Type', fontsize=12)
axes[0].set_xlabel('')
axes[0].set_ylabel('Number of Listings')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(room_counts):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontsize=9)

axes[1].pie(
    room_counts, labels=room_counts.index,
    autopct='%1.1f%%', startangle=90,
    colors=sns.color_palette('Oranges_d', len(room_counts))
)
axes[1].set_title('Market Share by Room Type', fontsize=12)

plt.suptitle('Bangkok — Room Type Composition', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 3. Availability vs Occupancy by Room Type
Bangkok median availability = 318 days (vs Singapore 357).  
Bangkok shows meaningfully higher estimated occupancy (~13% vs ~2%) — a structurally more active market.

In [ ]:
avail_stats = (
    df.groupby('room_type')
    .agg(
        median_availability  = ('availability_365', 'median'),
        median_occupancy_pct = ('occupancy_proxy', lambda x: round(x.median() * 100, 1)),
        listing_count        = ('id', 'count')
    )
    .sort_values('median_occupancy_pct', ascending=False)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = sns.color_palette('Oranges_d', len(avail_stats))

bars = axes[0].bar(avail_stats['room_type'], avail_stats['median_availability'], color=colors, edgecolor='white')
axes[0].set_title('Median Days Available per Year\n(lower = more likely in use)', fontsize=11)
axes[0].set_ylabel('Days Available (out of 365)')
axes[0].set_ylim(0, 365)
axes[0].axhline(y=365, color='red', linestyle='--', alpha=0.4, label='max (never booked)')
axes[0].legend(fontsize=8)
axes[0].tick_params(axis='x', rotation=30)
for bar, val in zip(bars, avail_stats['median_availability']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 4, f'{val:.0f}d', ha='center', fontsize=9)

bars2 = axes[1].bar(avail_stats['room_type'], avail_stats['median_occupancy_pct'], color=colors, edgecolor='white')
axes[1].set_title('Estimated Occupancy Rate by Room Type\n(proxy: 1 - availability/365)', fontsize=11)
axes[1].set_ylabel('Occupancy Estimate (%)')
axes[1].set_ylim(0, 60)
axes[1].tick_params(axis='x', rotation=30)
for bar, val in zip(bars2, avail_stats['median_occupancy_pct']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val}%', ha='center', fontsize=9)

plt.suptitle('Bangkok — Room Type: Availability & Occupancy', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print(avail_stats[['room_type','median_availability','median_occupancy_pct','listing_count']].to_string(index=False))

---
## 4. Neighbourhood — Median Price Ranking
Bangkok has 50 neighbourhoods vs Singapore's 44. Prices vary considerably — central districts (Watthana, Pathum Wan) typically command premiums.

In [ ]:
nb_price = (
    df.groupby('neighbourhood')['price']
    .agg(['median', 'count'])
    .rename(columns={'median': 'median_price', 'count': 'listing_count'})
    .query('listing_count >= 10')
    .sort_values('median_price', ascending=False)
    .head(20)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(
    nb_price['neighbourhood'][::-1],
    nb_price['median_price'][::-1],
    color=sns.color_palette('Oranges_d', len(nb_price)),
    edgecolor='white'
)
for bar, (_, row) in zip(bars, nb_price[::-1].iterrows()):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f'฿{row["median_price"]:,.0f}  (n={row["listing_count"]})',
            va='center', fontsize=8)

ax.set_title('Bangkok — Top 20 Neighbourhoods by Median Nightly Price\n(min 10 listings)', fontsize=12, pad=12)
ax.set_xlabel('Median Price (THB / night)')
ax.xaxis.set_major_formatter(mtick.StrMethodFormatter('฿{x:,.0f}'))
ax.set_xlim(0, nb_price['median_price'].max() * 1.3)
plt.tight_layout()
plt.show()

---
## 5. Neighbourhood — Listing Count vs Review Activity
Bangkok's top neighbourhoods by listing count are far larger than Singapore's, reflecting the 8x market size difference.

In [ ]:
nb_activity = (
    df.groupby('neighbourhood')
    .agg(
        listing_count     = ('id', 'count'),
        median_reviews_pm = ('reviews_per_month', 'median'),
        median_price      = ('price', 'median')
    )
    .query('listing_count >= 10')
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 7))
scatter = ax.scatter(
    nb_activity['listing_count'],
    nb_activity['median_reviews_pm'],
    c=nb_activity['median_price'],
    cmap='Oranges', s=80, alpha=0.8, edgecolors='grey', linewidths=0.4
)
plt.colorbar(scatter, ax=ax, label='Median Price (THB)')

top = nb_activity.nlargest(10, 'listing_count')
for _, row in top.iterrows():
    ax.annotate(row['neighbourhood'],
                (row['listing_count'], row['median_reviews_pm']),
                textcoords='offset points', xytext=(6, 2), fontsize=7.5)

ax.axvline(nb_activity['listing_count'].median(), color='gray', linestyle='--', alpha=0.5, linewidth=0.9)
ax.axhline(nb_activity['median_reviews_pm'].median(), color='gray', linestyle='--', alpha=0.5, linewidth=0.9)

ax.set_title('Bangkok — Neighbourhood: Listing Supply vs Review Activity\n(colour = median price)', fontsize=12, pad=12)
ax.set_xlabel('Number of Listings')
ax.set_ylabel('Median Reviews per Month')
plt.tight_layout()
plt.show()

---
## 6. Neighbourhood — Price vs Demand Side by Side

In [ ]:
nb_top20 = (
    df.groupby('neighbourhood')
    .agg(
        listing_count     = ('id', 'count'),
        median_price      = ('price', 'median'),
        median_reviews_pm = ('reviews_per_month', 'median')
    )
    .query('listing_count >= 10')
    .sort_values('listing_count', ascending=False)
    .head(20)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

nb_sorted_price = nb_top20.sort_values('median_price', ascending=True)
axes[0].barh(nb_sorted_price['neighbourhood'], nb_sorted_price['median_price'],
             color=sns.color_palette('Oranges_d', len(nb_sorted_price)), edgecolor='white')
axes[0].set_title('Median Price per Night\n(Top 20 by listing count)', fontsize=11)
axes[0].set_xlabel('Median Price (THB)')
axes[0].xaxis.set_major_formatter(mtick.StrMethodFormatter('฿{x:,.0f}'))

nb_sorted_rev = nb_top20.sort_values('median_reviews_pm', ascending=True)
axes[1].barh(nb_sorted_rev['neighbourhood'], nb_sorted_rev['median_reviews_pm'],
             color=sns.color_palette('Blues_d', len(nb_sorted_rev)), edgecolor='white')
axes[1].set_title('Median Reviews per Month\n(proxy for booking demand)', fontsize=11)
axes[1].set_xlabel('Median Reviews / Month')

plt.suptitle('Bangkok — Neighbourhood Price vs Demand Activity', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 7. Host Portfolio Segmentation

In [ ]:
df['host_tier'] = pd.cut(
    df['calculated_host_listings_count'],
    bins=[0, 1, 5, float('inf')],
    labels=['Single (1)', 'Small (2–5)', 'Commercial (6+)']
)

host_seg = (
    df.groupby('host_tier', observed=True)
    .agg(
        host_count    = ('host_id', 'nunique'),
        listing_count = ('id', 'count'),
        median_price  = ('price', 'median')
    )
    .reset_index()
)
host_seg['pct_listings'] = (host_seg['listing_count'] / host_seg['listing_count'].sum() * 100).round(1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
palette = ['#FF9800', '#FFB74D', '#FFE0B2']

axes[0].bar(host_seg['host_tier'], host_seg['host_count'], color=palette, edgecolor='white')
axes[0].set_title('Number of Hosts per Tier', fontsize=11)
axes[0].set_ylabel('Host Count')
axes[0].tick_params(axis='x', rotation=20)
for i, v in enumerate(host_seg['host_count']):
    axes[0].text(i, v + 10, f'{v:,}', ha='center', fontsize=9)

axes[1].bar(host_seg['host_tier'], host_seg['pct_listings'], color=palette, edgecolor='white')
axes[1].set_title('% of Total Listings Controlled', fontsize=11)
axes[1].set_ylabel('% of Listings')
axes[1].set_ylim(0, 100)
axes[1].tick_params(axis='x', rotation=20)
for i, v in enumerate(host_seg['pct_listings']):
    axes[1].text(i, v + 1, f'{v}%', ha='center', fontsize=9)

axes[2].bar(host_seg['host_tier'], host_seg['median_price'], color=palette, edgecolor='white')
axes[2].set_title('Median Price by Host Tier', fontsize=11)
axes[2].set_ylabel('Median Price (THB)')
axes[2].yaxis.set_major_formatter(mtick.StrMethodFormatter('฿{x:,.0f}'))
axes[2].tick_params(axis='x', rotation=20)
for i, v in enumerate(host_seg['median_price']):
    axes[2].text(i, v + 10, f'฿{v:,.0f}', ha='center', fontsize=9)

plt.suptitle('Bangkok — Host Portfolio Segmentation', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(host_seg[['host_tier','host_count','listing_count','pct_listings','median_price']].to_string(index=False))